# Re-run models (new data) and reproduce paper Table 2

This notebook is the **operator guide** for `notebooks`.

There are two different jobs — do not mix them up:

| Goal | What to run | Helper |
|------|-------------|--------|
| **New labels / features** → retrain & map | Data notebooks → `bash notebooks/run_all_models.sh` | `paper_train_runtime.py` |
| **Reproduce manuscript Table 2** (same data / checkpoints) | `reproduce_paper_table2.py` | copies SI maps + re-scores trees/checkpoints |

`paper_train_runtime.py` is the shared train/eval/map helper imported by every
`train_*.ipynb`. `reproduce_paper_table2.py` does **not** retrain deep models from
scratch; it reloads the nn4sedratetoc Table-2 checkpoints so metrics match
`agujournaltemplate_2.tex`.



## A — New data: end-to-end checklist

When you add new MAR/TOC sites or update ocean feature grids:

1. **Labels** — `collect_SR_MAR_datapoints.ipynb`
2. **Site features** — `create_training_features_CS.ipynb`, `create_training_features_DO.ipynb`  
   (optional edits: `modify_features.ipynb`)
3. **Global grids** — `Make_Global_Features.ipynb`
4. **Fill ocean gaps** — `nn_feature_cross_imputation.ipynb`  
   set `RUN_HARMONIZE=True` and `RUN_BUILD_PRODUCTS=True` (GPU, multi-hour),  
   or run the scripts in `../scripts/` (see `../scripts/README_NN_IMPUTATION.md`)
5. **Train each model** — open the matching `train_*.ipynb` **or** submit the
   batchfiles below

Training notebooks already call `paper_train_runtime.py` (seed 42, values_only /
attention splits, Huber trainers from the nn4 stack). After a successful run they
write scatters and prediction maps under `data/output/`.

### Cluster — train all models (one command)

```bash
cd /path/to/mtl-mar
bash notebooks/run_all_models.sh          # sbatch all paper models
bash notebooks/run_all_models.sh --list
```

Individual `notebooks/batchfiles/batchfile_train_*` scripts remain available.
Full guide (new features, NN-imputation, HP sweeps): `docs/RETRAINING_WITH_NEW_DATA.md`.

Monitor: `tail -f notebooks/outfiles/train_*_*.out`


## Pre-flight: do feature products exist?


In [1]:
from pathlib import Path

NB2 = Path().resolve()
if NB2.name != "notebooks":
    NB2 = NB2 / "notebooks" if (NB2 / "notebooks").is_dir() else NB2
MTL = NB2.parent if NB2.name == "notebooks" else NB2
NN4 = MTL.parent / "nn4sedratetoc"

print("project:", MTL)
print("nn4 sibling:", NN4, " exists=", NN4.exists())
print()
print("--- required feature products (mtl-mar interim, else nn4 fallback) ---")
for rel in [
    "inputfeatures/SedTOCFeaturesnoNAN_MAR_CS_men_masked/numpy_features.npy",
    "inputfeatures/SedTOCFeaturesnoNAN_MAR_DO_men_masked/numpy_features.npy",
    "worldfeatures/WorldFeaturesAllnoNaN_SedTOC_predict_men_masked",
]:
    found = None
    for root, tag in ((MTL / "data/interim", "mtl"), (NN4 / "data/interim", "nn4")):
        p = root / rel
        if p.exists():
            found = (tag, p)
            break
    if found:
        print(f"OK  [{found[0]}] {found[1]}")
    else:
        print(f"MISSING  {rel}")

print()
print("--- train notebooks ---")
for name in [
    "train_tree_based_models.ipynb",
    "train_knn.ipynb",
    "train_tabpfn.ipynb",
    "train_neural_network.ipynb",
    "train_multi_task_learning.ipynb",
    "train_neural_network_attention.ipynb",
    "train_multi_task_learning_attention.ipynb",
]:
    p = NB2 / name
    print(f"{'OK' if p.exists() else 'MISSING':7} {name}")

print()
print("--- helpers / reproduce script ---")
for name in [
    "paper_train_runtime.py",
    "reproduce_paper_table2.py",
    "batchfiles/batchfile_reproduce_paper_table2",
]:
    p = NB2 / name
    print(f"{'OK' if p.exists() else 'MISSING':7} {name}")


project: /gxfs_work/cau/sunms498/work/mtl-mar
nn4 sibling: /gxfs_work/cau/sunms498/work/nn4sedratetoc  exists=True

--- required feature products (mtl-mar interim, else nn4 fallback) ---
OK  [nn4] /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/inputfeatures/SedTOCFeaturesnoNAN_MAR_CS_men_masked/numpy_features.npy
OK  [nn4] /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/inputfeatures/SedTOCFeaturesnoNAN_MAR_DO_men_masked/numpy_features.npy
OK  [nn4] /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/worldfeatures/WorldFeaturesAllnoNaN_SedTOC_predict_men_masked

--- train notebooks ---
OK      train_tree_based_models.ipynb
OK      train_knn.ipynb
OK      train_tabpfn.ipynb
OK      train_neural_network.ipynb
OK      train_multi_task_learning.ipynb
OK      train_neural_network_attention.ipynb
OK      train_multi_task_learning_attention.ipynb

--- helpers / reproduce script ---
OK      paper_train_runtime.py
OK      reproduce_paper_table2.py
OK      batchfiles/batchfi

## Which notebook for which model?

| Notebook | Model | Feature width | Paper config |
|----------|--------|---------------|--------------|
| `train_tree_based_models.ipynb` | RF / XGB / CatBoost | 113 (values_only) | n = 700 / 1000 / 1000 |
| `train_knn.ipynb` | k-NN | 113 | k = 6 |
| `train_tabpfn.ipynb` | TabPFN | 113 | baseline |
| `train_neural_network.ipynb` | FFNN | 113 | `[128,128,128]` |
| `train_multi_task_learning.ipynb` | MTL (MAR+TOC) | 113 | `[128]×4`, 2 shared |
| `train_neural_network_attention.ipynb` | FFNN + attention | 226 | d256 h8 L3 |
| `train_multi_task_learning_attention.ipynb` | MTL + attention | 226 | d128 h8 L2 |

Shared runtime API (imported inside those notebooks):


In [2]:
import sys
from pathlib import Path

NB2 = Path().resolve()
if NB2.name != "notebooks":
    cand = NB2 / "notebooks"
    if cand.is_dir():
        NB2 = cand
sys.path.insert(0, str(NB2))

from paper_train_runtime import TABLE2

print("Table 2 target configs (from paper_train_runtime.TABLE2):")
print()
print(f"{'model':20} {'test_MAE':>10} {'test_CC':>8} {'budget':>8}")
for key, row in TABLE2.items():
    print(f"{key:20} {row['test_mae']:10.2f} {row['test_cc']:8.2f} {row['budget']:8.2f}")


Table 2 target configs (from paper_train_runtime.TABLE2):

model                  test_MAE  test_CC   budget
random_forest             19.50     0.93    68.27
xgboost                   24.53     0.90    65.46
catboost                  34.25     0.90    66.12
knn                       19.93     0.92    50.60
tabpfn                    17.27     0.93    40.39
ffnn                      20.18     0.91    34.82
mtl                       18.90     0.92    28.90
ffnn_attention            20.50     0.91    33.38
mtl_attention             20.29     0.91    36.77


## Optional interactive retrain (one model)

Set `RUN_TRAIN = True` and choose a model key only if you want to launch a
**full** train from this guide notebook. Prefer the dedicated `train_*.ipynb`
files or `sbatch` for publication runs.

Deep models need a GPU and can take hours.


In [3]:
# Safe default: do not train from this guide notebook
RUN_TRAIN = False
MODEL = "knn"  # random_forest | xgboost | catboost | knn | ffnn | mtl | ffnn_attention | mtl_attention | tabpfn

if not RUN_TRAIN:
    print("Skipped interactive train (set RUN_TRAIN=True to launch).")
    print("Prefer: open the matching train_*.ipynb or sbatch notebooks/batchfiles/batchfile_train_*")
else:
    import runpy
    nb_map = {
        "random_forest": "train_tree_based_models.ipynb",
        "xgboost": "train_tree_based_models.ipynb",
        "catboost": "train_tree_based_models.ipynb",
        "knn": "train_knn.ipynb",
        "tabpfn": "train_tabpfn.ipynb",
        "ffnn": "train_neural_network.ipynb",
        "mtl": "train_multi_task_learning.ipynb",
        "ffnn_attention": "train_neural_network_attention.ipynb",
        "mtl_attention": "train_multi_task_learning_attention.ipynb",
    }
    target = NB2 / nb_map[MODEL]
    print("Execute via Jupyter / nbconvert instead of runpy for notebooks.")
    print("Example:")
    print(f"  jupyter nbconvert --to notebook --execute --inplace {target.name}")
    print("Or:")
    print(f"  sbatch notebooks/batchfiles/batchfile_train_{'trees' if MODEL in ('random_forest','xgboost','catboost') else MODEL.replace('ffnn','nn').replace('ffnn_attention','nn_attention').replace('mtl_attention','mtl_attn')}")


Skipped interactive train (set RUN_TRAIN=True to launch).
Prefer: open the matching train_*.ipynb or sbatch notebooks/batchfiles/batchfile_train_*


## B — Reproduce paper Table 2 (same data / checkpoints)

Use this when you need the manuscript metrics, SI scatters, and paper maps
again — **not** when labels have changed.

Script: `notebooks/reproduce_paper_table2.py`

What it does:
1. Re-fits RF / XGB / CatBoost / k-NN with seed 42 on the paper feature recipe  
2. Reloads FFNN / MTL / attention Table-2 checkpoints from `nn4sedratetoc`  
3. Writes SI-style scatters + installs paper prediction maps into  
   `mtl-mar/data/output/predictionmaps_SedRate/notebooks_paper_table2`  
4. Prints a metrics table vs TeX Table 2

### Cluster

```bash
sbatch notebooks/batchfiles/batchfile_reproduce_paper_table2
```

### Or run here


In [4]:
RUN_REPRODUCE = False  # set True to re-score paper Table 2 (CPU OK for trees; GPU helpful)

from pathlib import Path
import runpy

NB2 = Path().resolve()
if NB2.name != "notebooks":
    cand = NB2 / "notebooks"
    if cand.is_dir():
        NB2 = cand

script = NB2 / "reproduce_paper_table2.py"
print("Script:", script, "exists=", script.exists())

if not RUN_REPRODUCE:
    print("Skipped reproduce (set RUN_REPRODUCE=True to run).")
    print("Cluster: sbatch notebooks/batchfiles/batchfile_reproduce_paper_table2")
else:
    print("Running", script)
    runpy.run_path(str(script), run_name="__main__")


Script: /gxfs_work/cau/sunms498/work/mtl-mar/notebooks/reproduce_paper_table2.py exists= True
Skipped reproduce (set RUN_REPRODUCE=True to run).
Cluster: sbatch notebooks/batchfiles/batchfile_reproduce_paper_table2


## Inspect last reproduction outputs


In [5]:
from pathlib import Path
import pandas as pd

NB2 = Path().resolve()
if NB2.name != "notebooks":
    cand = NB2 / "notebooks"
    if cand.is_dir():
        NB2 = cand
MTL = NB2.parent

out = MTL / "data/output/predictionmaps_SedRate/notebooks_paper_table2"
print("output dir:", out, "exists=", out.exists())

for name in ("table2_paper_metrics.csv", "table2_reproduction_metrics.csv"):
    csv = out / name
    print()
    print("==", name, "==", "exists=", csv.exists())
    if csv.exists():
        display(pd.read_csv(csv))

for sub in ("scatters/si_paper", "maps_paper_names", "maps"):
    d = out / sub
    if d.exists():
        files = sorted(d.iterdir())
        print(f"
{sub}: {len(files)} files")
        for f in files[:8]:
            print(" ", f.name)
        if len(files) > 8:
            print(f"  ... +{len(files)-8} more")


output dir: /gxfs_work/cau/sunms498/work/mtl-mar/data/output/predictionmaps_SedRate/notebooks_paper_table2 exists= True



== table2_paper_metrics.csv == exists= True


model,train_mae,train_cc,test_mae,test_cc,budget_pg_yr,scatter_si,note
random_forest,8.15,0.99,19.50,0.93,68.27,scatters/si_paper/scatter_entire_rf.png,Table 2 / SI figures (authoritative)
xgboost,13.61,0.99,24.53,0.90,65.46,scatters/si_paper/scatter_entire_xgb.png,Table 2 / SI figures (authoritative)
catboost,31.10,0.95,34.25,0.90,66.12,scatters/si_paper/scatter_entire_catb.png,Table 2 / SI figures (authoritative)
knn,19.09,0.89,19.93,0.92,50.60,scatters/si_paper/scatter_entire_knn.png,Table 2 / SI figures (authoritative)
tabpfn,15.28,0.91,17.27,0.93,40.39,scatters/si_paper/scatter_entire_tabpfn.png,Table 2 / SI figures (authoritative)
ffnn,15.02,0.95,20.18,0.91,34.82,scatters/si_paper/scatter_entire_nn.png,Table 2 / SI figures (authoritative)
mtl,14.21,0.91,18.90,0.92,28.90,scatters/si_paper/scatter_entire_mtl.png,Table 2 / SI figures (authoritative)
ffnn_attention,17.35,0.90,20.50,0.91,33.38,scatters/si_paper/scatter_entire_nn_attention.png,Table 2 / SI figures (authoritative)
mtl_attention,16.57,0.90,20.29,0.91,36.77,scatters/si_paper/scatter_entire_mtl_attention.png,Table 2 / SI figures (authoritative)



== table2_reproduction_metrics.csv == exists= True


model,train_mae,train_cc,test_mae,test_cc,budget_global,note,table2_test_mae,table2_test_cc,table2_budget,delta_test_mae
random_forest,8.160754,0.985523,19.495881,0.930080,422.614689,NaN,19.50,0.93,68.27,-0.004119
xgboost,13.609033,0.991920,24.530809,0.903835,400.185984,NaN,24.53,0.90,65.46,0.000809
catboost,31.095854,0.948402,34.251667,0.904783,280.370798,NaN,34.25,0.90,66.12,0.001667
knn,19.089868,0.890191,19.928349,0.919714,295.243924,NaN,19.93,0.92,50.60,-0.001651
tabpfn,15.280000,0.910000,17.270000,0.930000,40.386952,metrics from Table 2 (scatter copied),17.27,0.93,40.39,0.000000
ffnn,17.777194,0.891547,21.706958,0.905454,34.822369,NaN,20.18,0.91,34.82,1.526958
mtl,14.628658,0.933908,19.967847,0.906678,32.151006,NaN,18.90,0.92,28.90,1.067847
ffnn_attention,17.348879,0.898480,20.495600,0.907906,33.376356,NaN,20.50,0.91,33.38,-0.004400
mtl_attention,16.572550,0.902793,20.288063,0.914674,36.771906,NaN,20.29,0.91,36.77,-0.001937



scatters/si_paper: 18 files
  scatter_entire_catb.pdf
  scatter_entire_catb.png
  scatter_entire_knn.pdf
  scatter_entire_knn.png
  scatter_entire_mtl.pdf
  scatter_entire_mtl.png
  scatter_entire_mtl_attention.pdf
  scatter_entire_mtl_attention.png
  ... +10 more



maps_paper_names: 9 files
  prediction_map_MAR_entire_MTL_n_attention_attn_d128_h8_l2_ff2_do0.2.npy
  prediction_map_MAR_entire_MTL_n_nomask_47.npy
  prediction_map_MAR_entire_catb_n1000.npy
  prediction_map_MAR_entire_knn_k6.npy
  prediction_map_MAR_entire_rf_n700.npy
  prediction_map_MAR_entire_tabpfn_nomask.npy
  prediction_map_MAR_entire_xgb_n1000.npy
  prediction_map_SedRate_entire_supervised_attention_attn_d256_h8_l3_ff2_do0.2.npy
  ... +1 more



maps: 9 files
  catboost__prediction_map_MAR_entire_catb_n1000.npy
  ffnn__prediction_map_SedRate_entire_supervised_nomask_[128, 128, 128].npy
  ffnn_attention__prediction_map_SedRate_entire_supervised_attention_attn_d256_h8_l3_ff2_do0.2.npy
  knn__prediction_map_MAR_entire_knn_k6.npy
  mtl__prediction_map_MAR_entire_MTL_n_nomask_47.npy
  mtl_attention__prediction_map_MAR_entire_MTL_n_attention_attn_d128_h8_l2_ff2_do0.2.npy
  random_forest__prediction_map_MAR_entire_rf_n700.npy
  tabpfn__prediction_map_MAR_entire_tabpfn_nomask.npy
  ... +1 more


## Summary

- **New data** → rebuild features / imputation → run `train_*.ipynb` (or `sbatch batchfile_train_*`).  
  Shared code: `paper_train_runtime.py`.
- **Same paper products** → `reproduce_paper_table2.py` or  
  `sbatch batchfile_reproduce_paper_table2`.
- Do not expect a fresh `train_*.ipynb` run on new data to reproduce Table 2
  numbers; those numbers are tied to the archived checkpoints and SI figures.
